# EIA-861M Monthly Electricity Data

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

FILE_OLD = "data/raw/HS861M 1990-2009.xlsx"
FILE_NEW = "data/raw/HS861M 2010-.xlsx"

In [15]:
# Both files have a 3-row merged header; skip to the third row for column names
def load_eia861m(path):
    df = pd.read_excel(path, sheet_name=0, header=2)
    return df

old = load_eia861m(FILE_OLD)
new = load_eia861m(FILE_NEW)

print("Old shape:", old.shape)
print("New shape:", new.shape)
# old.head(3)

# Rename the first four columns consistently across both files
def rename_base_cols(df):
    cols = list(df.columns)
    cols[:4] = ["year", "month", "state", "data_status"]
    df.columns = cols
    return df

old = rename_base_cols(old)
new = rename_base_cols(new)

# Stack them
df = pd.concat([old, new], ignore_index=True)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

print("Combined shape:", df.shape)
print("Year range:", df['year'].min(), "-", df['year'].max())

price_cols = {"year": df.columns[0],
              "month": df.columns[1],
              "state": df.columns[2],
              "data_status": df.columns[3],
              "res_price": df.columns[7],
              "com_price": df.columns[11],
              "ind_price": df.columns[15],
              "total_price": df.columns[-1]}

df = df[list(price_cols.values())].copy()
df.columns = list(price_cols.keys())
# df.head()

# EIA uses '.' for missing. replace with NaN then convert price cols to float
price_col_names = ["res_price", "com_price", "ind_price", "total_price"]

df[price_col_names] = df[price_col_names].replace(".", np.nan)
df[price_col_names] = df[price_col_names].apply(pd.to_numeric, errors="coerce")

print("Missing values per column:")
print(df[price_col_names].isna().sum())

STATES = ["VA", "MD", "NC", "TX", "CA", "US"]

df_states = df[df["state"].isin(STATES)].copy()

print("Filtered shape:", df_states.shape)
print("States included:", sorted(df_states["state"].unique()))

df_states["date"] = pd.to_datetime(
    df_states[["year", "month"]].assign(day=1)
)

df_states = df_states.sort_values(["state", "date"]).reset_index(drop=True)
df_states.head()

# Check for duplicate state/month/year combos
dupes = df_states.duplicated(subset=["state", "year", "month"])
print(f"Duplicate rows: {dupes.sum()}")

Old shape: (12241, 28)
New shape: (10201, 24)
Combined shape: (22440, 28)
Year range: 1990 - 2026
Missing values per column:
res_price          0
com_price         13
ind_price         13
total_price    10200
dtype: int64
Filtered shape: (2200, 8)
States included: ['CA', 'MD', 'NC', 'TX', 'VA']
Duplicate rows: 0


In [16]:
OUT = "data/clean/monthly_eia861m_clean.csv"
df_states.to_csv(OUT, index=False)
print(f"Saved {len(df_states):,} rows to {OUT}")

Saved 2,200 rows to data/clean/monthly_eia861m_clean.csv
